In [6]:
%%HTML
<style>
    body {
        --vscode-font-family: "Fira Sans"
    }
</style>

# Python Concurrency: Multiprocessing Tutorial

Multiprocessing is used for **CPU-bound tasks** (e.g., heavy mathematical computations, image processing). It bypasses the Global Interpreter Lock (GIL) by creating separate OS processes, each with its own Python interpreter and memory space.

## 1. Multi-core Execution

In Python, `threading` is limited to a single core because of the GIL. `multiprocessing` allows you to utilize all available CPU cores by spawning multiple processes. Each process runs independently on a core, processing data in parallel.

**Note for macOS/Windows**: These systems use the `spawn` method to start processes, which means the main module must be importable. Therefore, you **must** wrap your entry point in `if __name__ == '__main__':`.

In [ ]:
import multiprocessing
import os

def info(title):
    print(f"{title} node name: {os.uname().nodename}")
    print(f"process id: {os.getpid()}")

def f(name):
    info('function f')
    print(f"hello {name}")

if __name__ == '__main__':
    info('main line')
    p = multiprocessing.Process(target=f, args=('bob',))
    p.start()
    p.join()

## 2. Process Concurrency Primitives

While processes have separate memory, you still need synchronization primitives when they access shared external resources (like a file) or shared memory segments.

### multiprocessing.Lock

#### Implementation Differences: `threading.Lock` vs. `multiprocessing.Lock` 

| Feature | `threading.Lock` | `multiprocessing.Lock` |
| :--- | :--- | :--- |
| **Implementation** | Uses OS-level thread primitives (e.g., **pthread mutex** on Unix, **CriticalSection** on Windows). | Uses system-wide primitives (e.g., **POSIX semaphores** on Unix, **named mutexes** on Windows). |
| **Memory** | Stored in the process's **shared memory space** reachable by all threads. | Stored in **isolated memory spaces**; state is managed by the **OS kernel** or through shared memory handles. |
| **Performance** | Very fast; minimal overhead for intra-process locking. | Slower; requires kernel calls and coordination across process boundaries. |
| **Scope** | Limited to threads within a **single process**. | Can be shared and used across **multiple independent processes**. |

- **Use Case**: Mutual exclusion for shared resources (e.g., writing to a single log file).
- **Example**: Preventing processes from garbling output to stdout.

In [ ]:
from multiprocessing import Process, Lock

def f(l, i):
    l.acquire()
    try:
        print('hello world', i)
    finally:
        l.release()

if __name__ == '__main__':
    lock = Lock()
    for num in range(10):
        Process(target=f, args=(lock, num)).start()

### multiprocessing.Event & Condition
- **Event**: Similar to `threading.Event`, used for binary signaling.
- **Condition**: Used for complex state-based synchronization (e.g. wait until a specific condition is met in shared memory).

## 3. Inter-Process Communication (IPC)

Processes do not share memory by default. Python provides several ways to move data between them.

### multiprocessing.Queue
- **Use Case**: Reliable communication between producers and consumers. Thread and process safe.
- **Mechanism**: Built on top of pipes and locks/semaphores.

In [ ]:
from multiprocessing import Process, Queue

def f(q):
    q.put([42, None, 'hello'])

if __name__ == '__main__':
    q = Queue()
    p = Process(target=f, args=(q,))
    p.start()
    print(q.get())    # prints "[42, None, 'hello']"
    p.join()

### multiprocessing.Pipe
- **Use Case**: Faster than Queue when you only need a connection between **two** processes.
- **Mechanism**: Returns a pair of connection objects connected by a duplex (two-way) pipe.

In [ ]:
from multiprocessing import Process, Pipe

def f(conn):
    conn.send([42, None, 'hello'])
    conn.send([42, None, 'hello2'])
    print(f"child received: {conn.recv()}")
    conn.close()

if __name__ == '__main__':
    parent_conn, child_conn = Pipe()
    p = Process(target=f, args=(child_conn,))
    p.start()
    parent_conn.send([42, None, 'from parent'])
    print(f"parent received: {parent_conn.recv()}")
    print(f"parent received: {parent_conn.recv()}")
    p.join()

### Shared Memory: Value and Array

#### How do processes share memory if they have separate memory spaces?

While it's true that processes have isolated **virtual memory spaces**, the Operating System provides a mechanism to create **Shared Memory Segments**. 

1. **OS-Level Allocation**: The kernel allocates a block of physical RAM as a shared segment.
2. **Memory Mapping**: Both processes "map" this physical block into their own virtual address space using system calls (like `shmget`/`shmat` on Unix or `CreateFileMapping`/`MapViewOfFile` on Windows).
3. **Direct Access**: Once mapped, changes made by one process at a virtual address are immediately visible to the other process at its own mapped address, as they point to the same physical RAM.

`multiprocessing.Value` and `multiprocessing.Array` are high-level wrappers that handle this memory mapping and wrap the raw memory in C-style types (`ctypes`).

**Note**: Because multiple processes can read/write simultaneously, you should usually use a **Lock** with shared memory objects to prevent race conditions.

- **Use Case**: When you need to share small amounts of data efficiently without the overhead of serialization.
- **Mechanism**: Uses C-style types in shared memory.

In [ ]:
from multiprocessing import Process, Value, Array

def f(n, a):
    n.value = 3.1415927
    for i in range(len(a)):
        a[i] = -a[i]

if __name__ == '__main__':
    num = Value('d', 0.0) # 'd' is double
    arr = Array('i', range(10)) # 'i' is integer

    p = Process(target=f, args=(num, arr))
    p.start()
    p.join()

    print(num.value)
    print(arr[:])

## 4. Important: The `join()` Requirement and macOS Semaphores

### Why you MUST use `p.join()`
In multiprocessing, it is critical that the main process stays alive until all child processes have finished their work. This is especially true on **macOS** (and some other Unix-like systems) due to how synchronization primitives like `Lock` are implemented.

### The macOS `FileNotFoundError` Issue
On macOS, `multiprocessing.Lock` is implemented using **named POSIX semaphores**. When the main process exits, it cleans up all the synchronization primitives it created. 

If you don't call `p.join()`, the main process might finish and exit while the child processes are still starting up or running. When a child process tries to "rebuild" the lock object (which it needs to do during the `spawn` process), it looks for the named semaphore in the OS. If the main process has already exited and cleaned it up, the child process will raise a `FileNotFoundError`.

### Best Practice
Always track your processes and call `join()` on them to ensure the main process remains alive and the resources persist until the work is done.

In [ ]:
from multiprocessing import Process, Lock

def worker(l, i):
    with l:
        print(f'Hello from process {i}')

if __name__ == '__main__':
    lock = Lock()
    processes = []
    for i in range(5):
        p = Process(target=worker, args=(lock, i))
        processes.append(p)
        p.start()
    
    # CRITICAL: Join all processes
    for p in processes:
        p.join()

## 5. Low-Level Shared Memory: `mmap` 

The `mmap` module allows you to create **memory-mapped files** or **anonymous memory segments**. Unlike `Value` or `Array`, which are structured, `mmap` provides a raw, file-like interface to a block of bytes.

### How it works:
1. **Mapping**: The OS maps a file (or a segment of RAM) into the process's virtual address space.
2. **Anonymous Mapping**: By using `-1` as the file descriptor, you create a shared memory segment that is not associated with any file on disk.
3. **Zero Copies**: Data is written directly to the memory segment. Both processes see the same physical memory, so there's no need for expensive serialization (pickling) or copying.

### Comparison:
| Feature | `Value`/`Array` | `mmap` |
| :--- | :--- | :--- |
| **Level** | High-level wrapper | Low-level byte buffer |
| **Data Types** | Structured (ctypes) | Raw bytes |
| **Ease of Use** | Simple, pythonic | Requires manual offset management |
| **Performance** | Good | Excellent (near-zero overhead) |

In [ ]:
import mmap
import multiprocessing
import os
import time

def worker(m):
    # Read from the start of the buffer
    m.seek(0)
    print(f"[Process {os.getpid()}] Reading current value: {m.read(5).decode()}")
    
    # Write back to it
    m.seek(0)
    m.write(b"CHILD")
    print(f"[Process {os.getpid()}] Updated buffer to 'CHILD'")

if __name__ == '__main__':
    # Create anonymous shared memory (1024 bytes)
    shared_mem = mmap.mmap(-1, 1024)
    shared_mem.write(b"START")

    p = multiprocessing.Process(target=worker, args=(shared_mem,))
    p.start()
    p.join()

    shared_mem.seek(0)
    print(f"[Main Process] Final value: {shared_mem.read(5).decode()}")
    shared_mem.close()